# SheCraft embedding model comparison

Compare embedding models on the same structure-aware chunks and the same 39 draft questions from Notebook 3. The candidates are English models, matching the current questions. A hit requires the expected PDF and page. Use the results to pick a candidate for further experiments.


## 1. Setup

In [1]:
import gc
import importlib
import json
import sys
from pathlib import Path
from shutil import which
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for folder in (cwd, *cwd.parents)
     for candidate in (folder, folder / "ai-service")
     if (candidate / "src" / "rag" / "doc_rag").is_dir()
     and (candidate / "rag_documents").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from the SheCraft Jewelry workspace.")
SRC_DIR = PROJECT_ROOT / "src" / "rag"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import torch
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Embedding comparison needs {error.name}. Install the model dependencies in the SheCraft notebook kernel, then rerun."
    ) from error

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", DEVICE)

c:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\tmp\shecraft-notebooks\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding device: cpu


## 2. Fix the chunking settings and select embedding models


In [2]:
CHUNKING_STRATEGY = "structure_aware"
CHUNKING_OPTIONS = {"chunk_size": 800, "overlap": 100}
MODEL_NAMES = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "intfloat/e5-small-v2",
    "BAAI/bge-small-en-v1.5",
]
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
BATCH_SIZE = 8
TOP_K = 5
ALLOW_MODEL_DOWNLOAD = True  # Fetch missing models on the first run.

print("Chunking:", CHUNKING_STRATEGY, CHUNKING_OPTIONS)
print("Embedding models:", MODEL_NAMES)


Chunking: structure_aware {'chunk_size': 800, 'overlap': 100}
Embedding models: ['sentence-transformers/all-MiniLM-L6-v2', 'intfloat/e5-small-v2', 'BAAI/bge-small-en-v1.5']


## 3. Extract documents and load the draft case set


In [3]:
extraction_module = importlib.import_module("doc_rag.2_extraction")
chunking_module = importlib.import_module("doc_rag.4_chunking")
embedding_module = importlib.import_module("doc_rag.5_embeddings")
extract_pdf = extraction_module.extract_pdf
chunk_extracted_documents = chunking_module.chunk_extracted_documents

DOCUMENTS_DIR = PROJECT_ROOT / "rag_documents"
TITLE_ONLY_PDFS = {"birthstone_guide.pdf"}
PDF_FILES = sorted(path for path in DOCUMENTS_DIR.rglob("*.pdf") if path.name not in TITLE_ONLY_PDFS)
CASES_FILE = PROJECT_ROOT / "data" / "rag" / "experiments" / "chunking_evaluation" / "document_chunking_cases.json"
with CASES_FILE.open(encoding="utf-8") as file:
    evaluation_payload = json.load(file)
evaluation_cases = evaluation_payload["cases"]
if evaluation_payload.get("status") != "draft":
    raise ValueError("Notebook 4 expects a provisional draft evaluation set.")

expected_files = {case["expected_file"] for case in evaluation_cases}
found_files = {path.name for path in PDF_FILES}
if expected_files != found_files:
    raise ValueError(f"Evaluation/document coverage mismatch. Missing PDFs: {sorted(expected_files - found_files)}; PDFs without cases: {sorted(found_files - expected_files)}")

extracted_documents = [extract_pdf(path, use_ocr_fallback=which("tesseract") is not None) for path in PDF_FILES]
chunks = chunk_extracted_documents(
    extracted_documents, strategy=CHUNKING_STRATEGY, **CHUNKING_OPTIONS
)
available_pages = set(zip(chunks["file_name"], chunks["page_number"]))
missing_labels = [
    (case["id"], case["expected_file"], page)
    for case in evaluation_cases
    for page in case["expected_pages"]
    if (case["expected_file"], page) not in available_pages
]
if not evaluation_cases or missing_labels:
    raise ValueError(f"Missing evaluation cases or labeled pages: {missing_labels}")
print(f"Fixed corpus: {len(chunks)} chunks, {len(expected_files)} PDFs")
print("Draft evaluation questions:", len(evaluation_cases))


Fixed corpus: 318 chunks, 39 PDFs
Draft evaluation questions: 39


## 4. Rank the same chunks with each embedding model


In [4]:
metric_rows = []
case_result_rows = []
model_failures = {}

for model_name in MODEL_NAMES:
    print("Evaluating:", model_name)
    try:
        model = SentenceTransformer(
            model_name, device=DEVICE, local_files_only=not ALLOW_MODEL_DOWNLOAD
        )
    except (OSError, RuntimeError, ValueError) as error:
        model_failures[model_name] = str(error)
        print(f"Could not load {model_name}: {error}")
        continue

    # Use the model's native token limit and the project's E5 query/passage prefixes.
    passages = embedding_module.prepare_texts(
        chunks["text"].tolist(), model_name=model_name, text_type="passage"
    )
    started = perf_counter()
    chunk_embeddings = embedding_module.embed_texts(passages, model, batch_size=BATCH_SIZE)
    queries = [case["query"] for case in evaluation_cases]
    if model_name == "BAAI/bge-small-en-v1.5":
        queries = [BGE_QUERY_PREFIX + query for query in queries]
    query_embeddings = embedding_module.embed_queries(
        queries,
        model=model, model_name=model_name, batch_size=BATCH_SIZE,
    )
    encoding_seconds = perf_counter() - started
    scores, positions = embedding_module.cosine_search(
        query_embeddings, chunk_embeddings, top_k=TOP_K
    )
    ranks = []
    for case_index, case in enumerate(evaluation_cases):
        expected_pages = set(case["expected_pages"])
        first_relevant_rank = None
        top_results = []
        for rank, position in enumerate(positions[case_index], start=1):
            chunk = chunks.iloc[int(position)]
            same_document = chunk["file_name"] == case["expected_file"]
            relevant = bool(same_document and int(chunk["page_number"]) in expected_pages)
            if relevant and first_relevant_rank is None:
                first_relevant_rank = rank
            top_results.append({
                "rank": rank, "chunk_id": chunk["chunk_id"], "file_name": chunk["file_name"],
                "page_number": int(chunk["page_number"]), "score": round(float(scores[case_index, rank - 1]), 6),
                "relevant": relevant,
            })
        ranks.append(first_relevant_rank)
        case_result_rows.append({
            "model": model_name, "case_id": case["id"], "query": case["query"],
            "expected_file": case["expected_file"], "expected_pages": case["expected_pages"],
            "answer_keywords": case.get("answer_keywords", []),
            "first_relevant_rank": first_relevant_rank, "top_5": top_results,
        })
    metric_rows.append({
        "model": model_name, "cases": len(ranks),
        "embedding_dimensions": int(chunk_embeddings.shape[1]),
        "max_input_tokens": model.max_seq_length,
        "encoding_seconds": round(encoding_seconds, 2),
        "recall_at_1": float(np.mean([rank is not None and rank <= 1 for rank in ranks])),
        "recall_at_3": float(np.mean([rank is not None and rank <= 3 for rank in ranks])),
        "recall_at_5": float(np.mean([rank is not None and rank <= 5 for rank in ranks])),
        "mrr_at_5": float(np.mean([1 / rank if rank is not None and rank <= 5 else 0 for rank in ranks])),
    })
    del model, chunk_embeddings, query_embeddings
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if not metric_rows:
    raise RuntimeError("None of the embedding models loaded. See the model errors above.")

model_metrics = pd.DataFrame(metric_rows).sort_values(
    ["recall_at_5", "mrr_at_5", "recall_at_1"], ascending=False
)
display(model_metrics)
case_results = pd.DataFrame(case_result_rows)
if model_failures:
    print("Comparison incomplete. Review the failed model loads and rerun before choosing.")


Evaluating: sentence-transformers/all-MiniLM-L6-v2


Batches: 100%|██████████| 40/40 [00:10<00:00,  3.68it/s]


Evaluating: intfloat/e5-small-v2


Batches: 100%|██████████| 40/40 [00:18<00:00,  2.14it/s]


Evaluating: BAAI/bge-small-en-v1.5


c:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\tmp\shecraft-notebooks\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USP\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 40/40 [00

,model,cases,embedding_dimensions,max_input_tokens,encoding_seconds,recall_at_1,recall_at_3,recall_at_5,mrr_at_5
1,intfloat/e5-small-v2,39,384,512,19.22,0.897436,0.974359,1.0,0.942308
2,BAAI/bge-small-en-v1.5,39,384,512,24.75,0.871795,1.000000,1.0,0.935897
0,sentence-transformers/all-MiniLM-L6-v2,39,384,256,11.23,0.794872,0.974359,1.0,0.881197


## 5. Review missed questions and compare per-question ranks


In [5]:
failures = case_results[case_results["first_relevant_rank"].isna()]
display(failures[["model", "case_id", "query", "expected_file", "expected_pages"]])
rank_comparison = case_results.pivot(
    index=["case_id", "query", "expected_file"], columns="model", values="first_relevant_rank"
).reset_index()
display(rank_comparison)


,model,case_id,query,expected_file,expected_pages


model,case_id,query,expected_file,BAAI/bge-small-en-v1.5,intfloat/e5-small-v2,sentence-transformers/all-MiniLM-L6-v2
0,bench_pave,"What is pavé setting, and what bench-work cons...",jewelry_settings_bench_guide.pdf,2,2,5
1,buying_gemstones,What should a buyer ask about gemstone treatme...,gemstone_diamond_pearl_buying_guide.pdf,1,1,1
2,chain_durability,Which chain styles are durable or difficult to...,jewelry_chain_guide.pdf,1,1,1
3,everyday_ring_metal,How should someone choose a ring metal for dai...,ring_metals_everyday_wear.pdf,1,1,1
4,gem_alexandrite,How does alexandrite change color under differ...,alexandrite.pdf,1,1,1
5,gem_amber,"What is amber, and what kinds of inclusions ca...",amber.pdf,1,1,1
6,gem_amethyst,"What is amethyst, what is its color, and how h...",amethyst.pdf,1,1,1
7,gem_ametrine,What makes ametrine different from amethyst or...,ametrine.pdf,1,1,1
8,gem_aquamarine,What gives aquamarine its name and characteris...,aquamarine.pdf,1,1,1
9,gem_citrine,"What colors can citrine have, and what causes ...",citrine.pdf,1,1,1


## 6. Save the draft embedding comparison


In [6]:
RESULTS_FILE = PROJECT_ROOT / "data" / "rag" / "experiments" / "embedding_model_comparison" / "results.json"
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "schema_version": 1,
    "evaluation_type": "provisional_notebook_technique_comparison",
    "status": "draft",
    "comparison_complete": not model_failures,
    "requested_models": MODEL_NAMES,
    "query_prefixes": {"BAAI/bge-small-en-v1.5": BGE_QUERY_PREFIX, "intfloat/e5-small-v2": "query: "},
    "model_load_failures": model_failures,
    "relevance": "expected PDF and page",
    "top_k": TOP_K,
    "device": DEVICE,
    "evaluation_case_file": str(CASES_FILE.relative_to(PROJECT_ROOT)),
    "chunking_strategy": CHUNKING_STRATEGY,
    "chunking_options": CHUNKING_OPTIONS,
    "chunk_count": len(chunks),
    "metrics": model_metrics.to_dict(orient="records"),
    "case_results": case_result_rows,
}
with RESULTS_FILE.open("w", encoding="utf-8") as file:
    json.dump(payload, file, ensure_ascii=False, indent=2)
print("Saved provisional results:", RESULTS_FILE)
print("Comparison complete:", not model_failures)


Saved provisional results: C:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\data\rag\experiments\embedding_model_comparison\results.json
Comparison complete: True
